In [1]:
import requests
from fake_useragent import UserAgent
import json
import urllib
from typing import List, Optional, Dict, Any
from pydantic import BaseModel
import time
import logging
import rich
import itertools

In [2]:
logging.basicConfig(level=logging.DEBUG)
logging.info("Logging Ready")

INFO:root:Logging Ready


In [3]:
headers: dict[str, str] = {
    "User-Agent": UserAgent().chrome
}
auth_session = requests.Session()
session = requests.Session()
auth_session.headers.update(headers)

In [4]:
login_response = auth_session.get("https://www.reserveamerica.com/signin")
login_response.raise_for_status()
login_response

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): www.reserveamerica.com:443
DEBUG:urllib3.connectionpool:https://www.reserveamerica.com:443 "GET /signin HTTP/1.1" 200 None


<Response [200]>

In [5]:
cookies: dict[str, str] = {
    "idToken": login_response.cookies["idToken"], 
    "a1Data": login_response.cookies["a1Data"], 
}
auth_headers: dict[str, str] = {
    "Authorization": login_response.cookies["idToken"], 
    "A1Data":  urllib.parse.unquote(login_response.cookies["a1Data"]), 
}
headers.update(auth_headers)
session.cookies.update(cookies)
session.headers.update(headers)

In [6]:
class Control(BaseModel):
    currentPage: int
    pageSize: int

class Agency(BaseModel):
    id: int
    code: str
    name: str
    logo: Optional[str] = None
    url: Optional[str] = None

class Affiliation(BaseModel):
    facilityID: int
    name: str
    url: Optional[str] = None
    agencies: List[Agency]

class Cta(BaseModel):
    caption: str
    url: Optional[str] = None
    type: str

class Coordinates(BaseModel):
    latitude: float
    longitude: float
    latitudeDec: Optional[float] = None
    latitudeDegMinSec: Optional[str] = None
    longitudeDec: Optional[float] = None
    longitudeDegMinSec: Optional[str] = None

class OfferingSupport(BaseModel):
    business: str
    type: str
    url: str

class Details(BaseModel):
    id: int
    contrCode: str
    stateCodes: List[str]
    name: str
    baseURL: Optional[str] = None
    imageURL: Optional[str] = None
    crowdriffId: Optional[str] = None
    verifiableAvailability: bool
    availability: Optional[Any] = None  # Always None in sample
    affiliation: Affiliation
    facilityTypes: List[str]
    description: Optional[str] = None
    flexResult: Optional[Any] = None
    cta: Cta
    dayUse: bool
    dayUseOnly: bool
    ratingSupport: bool
    rating: Optional[float] = None
    actvAdvInd: Optional[Any] = None
    favoriteSupport: bool
    favorite: bool
    webAddress: Optional[str] = None
    coordinates: Optional[Coordinates] = None
    hiddenOnMap: bool
    crossOverCustomLanding: bool
    legacyRequired: bool
    lineOfBusinesses: Optional[List[str]] = None
    offeringSupport: Optional[List[OfferingSupport]] = None
    nonClientFacility: bool

class Record(BaseModel):
    type: str
    namingId: str
    namingLabel: str
    id: int
    name: str
    targetItem: bool
    matchingFilters: bool
    reservableType: Optional[str] = None
    available: bool
    proximity: float
    summary: Optional[str] = None
    details: Details
    logoSrc: Optional[str] = None

class CampingResponse(BaseModel):
    totalRecords: int
    totalPages: int
    startIndex: int
    endIndex: int
    control: Control
    records: List[Record]
    typesSummary: Dict[str, int]
    actvAdvInfo: Optional[Any] = None

In [7]:
search_params = {"stype": "state", "tstc": "CO", "rcp": 0}
search_url = "https://api.reserveamerica.com/jaxrs-json/search"

In [8]:
responses: list[CampingResponse] = []
while True:
    search_response = session.get(search_url, params=search_params)
    search_response.raise_for_status()
    search = CampingResponse(**search_response.json())
    responses.append(search)
    if not search.records:
        break
    elif search.control.currentPage == (search.totalPages - 1):
        break
    else:
        search_params["rcp"] = search_params["rcp"] + 1
        time.sleep(1)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.reserveamerica.com:443
DEBUG:urllib3.connectionpool:https://api.reserveamerica.com:443 "GET /jaxrs-json/search?stype=state&tstc=CO&rcp=0 HTTP/1.1" 200 52633
DEBUG:urllib3.connectionpool:https://api.reserveamerica.com:443 "GET /jaxrs-json/search?stype=state&tstc=CO&rcp=1 HTTP/1.1" 200 53948
DEBUG:urllib3.connectionpool:https://api.reserveamerica.com:443 "GET /jaxrs-json/search?stype=state&tstc=CO&rcp=2 HTTP/1.1" 200 54925
DEBUG:urllib3.connectionpool:https://api.reserveamerica.com:443 "GET /jaxrs-json/search?stype=state&tstc=CO&rcp=3 HTTP/1.1" 200 56002
DEBUG:urllib3.connectionpool:https://api.reserveamerica.com:443 "GET /jaxrs-json/search?stype=state&tstc=CO&rcp=4 HTTP/1.1" 200 50721
DEBUG:urllib3.connectionpool:https://api.reserveamerica.com:443 "GET /jaxrs-json/search?stype=state&tstc=CO&rcp=5 HTTP/1.1" 200 53672
DEBUG:urllib3.connectionpool:https://api.reserveamerica.com:443 "GET /jaxrs-json/search?stype=state&tstc=

In [9]:
all_records = list(itertools.chain(*[item.records for item in responses]))
len(all_records)

339

In [10]:
rich.print(all_records[0])

Record(
    type='camping',
    namingId='FLST_755487',
    namingLabel='ALDER GUARD STATION',
    id=755487,
    name='ALDER GUARD STATION',
    targetItem=False,
    matchingFilters=True,
    reservableType=None,
    available=False,
    proximity=0.0,
    summary=None,
    details=Details(
        id=755487,
        contrCode='FLST',
        stateCodes=['CO'],
        name='ALDER GUARD STATION',
        baseURL=None,
        imageURL=None,
        crowdriffId=None,
        verifiableAvailability=False,
        availability=None,
        affiliation=Affiliation(
            facilityID=0,
            name='',
            url=None,
            agencies=[Agency(id=751960, code='USFS', name='USFS', logo=None, url=None)]
        ),
        facilityTypes=['camping'],
        description=None,
        flexResult=None,
        cta=Cta(caption='label_cta_check_availability', url=None, type='LISTING'),
        dayUse=True,
        dayUseOnly=True,
        ratingSupport=False,
        rating=None,
        actvAdvInd=None,
        favoriteSupport=True,
        favorite=False,
        webAddress=None,
        coordinates=Coordinates(
            latitude=37.7044444,
            longitude=-106.6466667,
            latitudeDec=None,
            latitudeDegMinSec=None,
            longitudeDec=None,
            longitudeDegMinSec=None
        ),
        hiddenOnMap=False,
        crossOverCustomLanding=False,
        legacyRequired=False,
        lineOfBusinesses=None,
        offeringSupport=None,
        nonClientFacility=True
    ),
    logoSrc=None
)